# Create JS-Struktur

In [58]:
import json
import requests
import xml.etree.ElementTree as ET

WMTS_URL = "https://wmts.geo.admin.ch/EPSG/2056/1.0.0/WMTSCapabilities.xml"
WMS_URL = "https://wms.geo.admin.ch/?SERVICE=WMS&VERSION=1.3.0&REQUEST=GetCapabilities"


# 1. Baue Mapping: layerBodId -> geocatId aus WMTS & WMS
import json
import requests
import xml.etree.ElementTree as ET

WMTS_URL = "https://wmts.geo.admin.ch/EPSG/2056/1.0.0/WMTSCapabilities.xml"
WMS_URL = "https://wms.geo.admin.ch/?SERVICE=WMS&VERSION=1.3.0&REQUEST=GetCapabilities"

# 1. Baue Mapping: layerBodId -> geocatId aus WMTS & WMS
def build_geocatid_mapping():
    def parse_capabilities(url, service_name):
        print(f"🔄 Lade {service_name} Capabilities...")
        resp = requests.get(url)
        resp.raise_for_status()
        content = resp.content

        if service_name == 'WMTS':
            ns = {
                'wmts': 'http://www.opengis.net/wmts/1.0',
                'ows': 'http://www.opengis.net/ows/1.1',
                'xlink': 'http://www.w3.org/1999/xlink'
            }
            root = ET.fromstring(content)
            layers = root.findall('.//wmts:Layer', ns)
            mapping = {}
            for layer in layers:
                identifier_elem = layer.find('ows:Identifier', ns)
                if identifier_elem is None:
                    continue
                bod_id = identifier_elem.text
                geocat_id = ""
                metadata = layer.find('ows:Metadata', ns)
                if metadata is not None:
                    href = metadata.attrib.get('{http://www.w3.org/1999/xlink}href')
                    if href and 'metadata/' in href:
                        geocat_id = href.split('metadata/')[-1].split('/')[0]
                if bod_id and geocat_id:
                    # ----
                    # mapping[bod_id] = geocat_id
                    mapping[bod_id] = {
                        "geocatId": geocat_id,
                        "source": service_name
                    }
                    # ----
            print(f"✅ {len(mapping)} WMTS Layer mit geocatId")
            return mapping

        elif service_name == 'WMS':
            ns = {
                'wms': 'http://www.opengis.net/wms',
                'xlink': 'http://www.w3.org/1999/xlink'
            }
            root = ET.fromstring(content)
            layers = root.findall('.//wms:Layer', ns)
            mapping = {}
            for layer in layers:
                name_elem = layer.find('wms:Name', ns)
                if name_elem is None:
                    continue
                bod_id = name_elem.text
                geocat_id = ""
                metadata_url = layer.find('wms:MetadataURL', ns)
                if metadata_url is not None:
                    online_resource = metadata_url.find('wms:OnlineResource', ns)
                    if online_resource is not None:
                        href = online_resource.attrib.get('{http://www.w3.org/1999/xlink}href')
                        if href and 'metadata/' in href:
                            geocat_id = href.split('metadata/')[-1].split('/')[0]
                if bod_id and geocat_id:
                    mapping[bod_id] = {
                        "geocatId": geocat_id,
                        "source": service_name
                    }
            print(f"✅ {len(mapping)} WMS Layer mit geocatId")
            return mapping


        else:
            print(f"❌ Unbekannter Service: {service_name}")
            return {}

    wmts_map = parse_capabilities(WMTS_URL, 'WMTS')
    wms_map = parse_capabilities(WMS_URL, 'WMS')

    # WMTS > WMS: Wenn WMTS keine geocatId hat, nimm WMS
    combined_map = wms_map.copy()
    combined_map.update(wmts_map)  # WMTS überschreibt WMS, wenn vorhanden
    return combined_map



# 2. Slugify für Labels
def slugify(text):
    return (
        text.lower()
        .replace("'", "")
        .replace('"', "")
        .replace("&", "and")
        .replace(",", "")
        .replace(".", "_")
        .replace("-", "_")
        .replace(" ", "_")
    )


# 3. JSON -> JS Konvertierung mit geocatId-Einbindung

# def convert_node_to_js(node, geocat_map, indent=2):
#     lines = []
#     pad = ' ' * indent
#     label = node.get("label", "no_label")

#     if node["category"] == "topic":
#         lines.append(f'{pad}// Topic - {label}')
#         lines.append(f'{pad}{{')
#         lines.append(f'{pad}  label: t(\'grp_{slugify(label)}_label\'),')
#         lines.append(f'{pad}  children: [')

#         for child in node.get("children", []):
#             lines += convert_node_to_js(child, geocat_map, indent + 4)
#             lines.append(f'{pad}    ,')

#         if lines[-1].strip() == ',':
#             lines.pop()
#         lines.append(f'{pad}  ]')
#         lines.append(f'{pad}}}')
#     # elif node["category"] == "layer":
#     #     layer_bod_id = node.get("layerBodId", "")
#     #     info = geocat_map.get(layer_bod_id, {})
#     #     geocat_id = info.get("geocatId", "")
#     #     source = info.get("source", "unknown")

#     #     if not geocat_id:
#     #         print(f"⚠️  Keine geocatId gefunden für Layer: {layer_bod_id}")

#     #     lines.append(f'{pad}// Layer - {label} - {source}')
#     #     lines.append(f'{pad}{{')
#     #     lines.append(f'{pad}  type: LayerType.swisstopoWMTS,')
#     #     lines.append(f'{pad}  label: t(\'lyr_{slugify(label)}_label\'),')
#     #     lines.append(f'{pad}  layer: \'{layer_bod_id}\',')
#     #     lines.append(f'{pad}  maximumLevel: 18,')
#     #     lines.append(f'{pad}  visible: false,')
#     #     lines.append(f'{pad}  displayed: false,')
#     #     lines.append(f'{pad}  opacity: 0.7,')
#     #     lines.append(f'{pad}  queryType: \'geoadmin\',')
#     #     lines.append(f'{pad}  geocatId: \'{geocat_id}\',')
#     #     lines.append(f'{pad}  legend: \'{layer_bod_id}\'')
#     #     lines.append(f'{pad}}}')

#     elif node["category"] == "layer":
#         layer_bod_id = node.get("layerBodId", "")
#         info = geocat_map.get(layer_bod_id, {})
#         geocat_id = info.get("geocatId", "")
#         source = info.get("source", "unknown")

#         if not geocat_id:
#             print(f"⚠️  Keine geocatId gefunden für Layer: {layer_bod_id}")

#         lines.append(f'{pad}// Layer - {layer_bod_id} - {source}')
#         lines.append(f'{pad}{{')
#         lines.append(f'{pad}  type: LayerType.swisstopoWMTS,')
#         lines.append(f'{pad}  label: t(\'lyr_{slugify(layer_bod_id)}_label\'),')
#         lines.append(f'{pad}  layer: \'{layer_bod_id}\',')
#         lines.append(f'{pad}  maximumLevel: 18,')
#         lines.append(f'{pad}  visible: false,')
#         lines.append(f'{pad}  displayed: false,')
#         lines.append(f'{pad}  opacity: 0.7,')
#         lines.append(f'{pad}  queryType: \'geoadmin\',')
#         lines.append(f'{pad}  geocatId: \'{geocat_id}\',')
#         lines.append(f'{pad}  legend: \'{layer_bod_id}\'')
#         lines.append(f'{pad}}}')


#     return lines

def convert_node_to_js(node, geocat_map, indent=2):
    lines = []
    pad = ' ' * indent
    category = node.get("category", "")
    label = node.get("label", "no_label")
    node_id = node.get("id", None)
    layer_bod_id = node.get("layerBodId", "")

    if category == "topic":
        # Key basierend auf id, falls vorhanden, sonst label fallback
        if node_id is not None:
            key = f"grp_{str(node_id)}_label"
        else:
            key = f"grp_{slugify(label)}_label"
        lines.append(f'{pad}// Topic - {label}')
        lines.append(f'{pad}{{')
        lines.append(f'{pad}  label: t(\'{key}\'),')
        lines.append(f'{pad}  children: [')

        for child in node.get("children", []):
            lines += convert_node_to_js(child, geocat_map, indent + 4)
            lines.append(f'{pad}    ,')

        if lines[-1].strip() == ',':
            lines.pop()
        lines.append(f'{pad}  ]')
        lines.append(f'{pad}}}')

    elif category == "layer":
        info = geocat_map.get(layer_bod_id, {})
        geocat_id = info.get("geocatId", "")
        source = info.get("source", "unknown")

        if not geocat_id:
            print(f"⚠️  Keine geocatId gefunden für Layer: {layer_bod_id}")

        key = f"lyr_{slugify(layer_bod_id if layer_bod_id else label)}_label"

        lines.append(f'{pad}// Layer - {label} - {source}')
        lines.append(f'{pad}{{')
        lines.append(f'{pad}  type: LayerType.swisstopoWMTS,')
        lines.append(f'{pad}  label: t(\'{key}\'),')
        lines.append(f'{pad}  layer: \'{layer_bod_id}\',')
        lines.append(f'{pad}  maximumLevel: 18,')
        lines.append(f'{pad}  visible: false,')
        lines.append(f'{pad}  displayed: false,')
        lines.append(f'{pad}  opacity: 0.7,')
        lines.append(f'{pad}  queryType: \'geoadmin\',')
        lines.append(f'{pad}  geocatId: \'{geocat_id}\',')
        lines.append(f'{pad}  legend: \'{layer_bod_id}\'')
        lines.append(f'{pad}}}')

    return lines



# 4. Hauptfunktion
def main():
    input_path = "./input/mga_layertree_en.json"
    output_path = "./output/layerTree.js"

    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    geocat_map = build_geocatid_mapping()

    # Zugriff auf root → children (direkte topics)
    root_children = data["results"]["root"]["children"]

    converted_blocks = [
        "// 🧾 Generated LayerTree Groups"
    ]

    for idx, topic_node in enumerate(root_children, start=1):
        if topic_node.get("category") != "topic":
            continue  # Nur echte Topics verarbeiten

        const_name = f"group_{idx:02d}"
        converted_lines = convert_node_to_js(topic_node, geocat_map, indent=2)
        block = [
            f"const {const_name}: LayerTreeNode = ",
            *converted_lines,
            ""
        ]
        converted_blocks.extend(block)

    # Alles in Datei schreiben
    with open(output_path, "w", encoding="utf-8") as f:
        f.write('\n'.join(converted_blocks))

    print(f"✅ JS-Datei erfolgreich erstellt: {output_path}")



if __name__ == "__main__":
    main()


🔄 Lade WMTS Capabilities...
✅ 662 WMTS Layer mit geocatId
🔄 Lade WMS Capabilities...
✅ 837 WMS Layer mit geocatId
✅ JS-Datei erfolgreich erstellt: ./output/layerTree.js


# Create translations

In [59]:
import json
import os

def slugify(text):
    return (
        text.lower()
        .replace("'", "")
        .replace('"', "")
        .replace("&", "and")
        .replace(",", "")
        .replace(".", "_")
        .replace("-", "_")
        .replace(" ", "_")
    )

# def extract_labels(nodes, lang_map, key_map=None):
#     """
#     Extracts labels from nodes and returns a dict:
#     key_map: key -> node (only from EN, to get keys from layerBodId or label)
#     lang_map: language -> dict key->label
#     """

#     for node in nodes:
#         category = node.get("category")
#         label = node.get("label", "")
#         layer_bod_id = node.get("layerBodId", "")

#         # Nur für topic oder layer
#         if category in ("topic", "layer"):
#             if category == "topic":
#                 # Key basiert auf Label
#                 key = f"grp_{slugify(label)}_label"
#             elif category == "layer":
#                 # Key basiert auf layerBodId, falls vorhanden, sonst label fallback
#                 base_for_key = layer_bod_id if layer_bod_id else label
#                 key = f"lyr_{slugify(base_for_key)}_label"

#             # key_map (nur in EN) speichern
#             if key_map is not None:
#                 key_map[key] = node

#             # labels speichern pro Sprache (lang_map)
#             # Das wird bei Aufruf außerhalb befüllt (siehe unten)
#             # Hier nur Key-Existenz sicherstellen
#             if key not in lang_map:
#                 lang_map[key] = label

#         # Rekursion
#         children = node.get("children", [])
#         if children:
#             extract_labels(children, lang_map, key_map)

def extract_labels(nodes, lang_map, key_map=None, is_en=False):
    for node in nodes:
        category = node.get("category")
        label = node.get("label", "")
        node_id = node.get("id", "")
        layer_bod_id = node.get("layerBodId", "")

        if category == "topic":
            # Key nur für Topics anhand der ID generieren
            if node_id:
                #key = f"grp_{slugify(node_id)}_label"
                key = f"grp_{slugify(str(node_id))}_label"
            else:
                key = f"grp_{slugify(label)}_label"

        elif category == "layer":
            base_for_key = layer_bod_id if layer_bod_id else label
            key = f"lyr_{slugify(base_for_key)}_label"

        else:
            continue

        if is_en and key_map is not None:
            key_map[key] = node

        if key not in lang_map:
            lang_map[key] = label

        children = node.get("children", [])
        if children:
            extract_labels(children, lang_map, key_map, is_en=is_en)



def generate_translation_files(input_dir, output_dir):
    lang_files = {
        "de": "mga_layertree_de.json",
        "fr": "mga_layertree_fr.json",
        "it": "mga_layertree_it.json",
        "en": "mga_layertree_en.json"
    }

    os.makedirs(output_dir, exist_ok=True)

    # 1. EN laden und Key-Map aufbauen
    en_path = os.path.join(input_dir, lang_files["en"])
    with open(en_path, "r", encoding="utf-8") as f:
        en_data = json.load(f)

    en_root_children = en_data.get("results", {}).get("root", {}).get("children", [])
    key_map = {}  # key -> node (für Keys)
    # extract_labels(en_root_children, {}, key_map=key_map)
    extract_labels(en_root_children, {}, key_map=key_map, is_en=True)

    # 2. Alle Sprachen laden, Labels extrahieren, Map aufbauen
    lang_label_maps = {}
    for lang, filename in lang_files.items():
        path = os.path.join(input_dir, filename)
        if not os.path.isfile(path):
            print(f"⚠️ Datei nicht gefunden: {path}")
            continue

        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        root_children = data.get("results", {}).get("root", {}).get("children", [])
        lang_map = {}
        #extract_labels(root_children, lang_map)
        extract_labels(root_children, lang_map, is_en=False)
        lang_label_maps[lang] = lang_map

    # 3. Für jede Sprache die Übersetzungen schreiben, basierend auf EN Keys
    for lang, lang_map in lang_label_maps.items():
        translations = {}
        for key in key_map.keys():
            # Wert aus dieser Sprache, wenn nicht vorhanden, fallback auf EN
            value = lang_map.get(key) or lang_label_maps["en"].get(key) or ""
            translations[key] = value

        out_path = os.path.join(output_dir, f"translations_{lang}.json")
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(translations, f, ensure_ascii=False, indent=2)

        print(f"✅ {len(translations)} Einträge in: {out_path}")

if __name__ == "__main__":
    generate_translation_files(input_dir="./input", output_dir="./output/translations")


✅ 150 Einträge in: ./output/translations/translations_de.json
✅ 150 Einträge in: ./output/translations/translations_fr.json
✅ 150 Einträge in: ./output/translations/translations_it.json
✅ 150 Einträge in: ./output/translations/translations_en.json
